# Quick Introduction to Generating Counterfactual Explanations using DiCE

Following the [official DiCE getting started notebook](https://github.com/interpretml/DiCE/blob/main/docs/source/notebooks/DiCE_getting_started.ipynb).

Exploring "what-if" scenarios is an important way to inspect a machine learning model. DiCE generates *counterfactuals* — data points that answer:

> Given that the model's output for input $x$ is $y$, what minimal change to $x$ would flip the output to a different class?

Good counterfactuals are:
- **Proximate** — close to the original input
- **Sparse** — change as few features as possible
- **Diverse** — a varied set of alternatives, not all the same
- **Feasible** — realistic values, not physically impossible combinations

DiCE supports two families of methods:
- **Model-agnostic** — works with any black-box classifier or regressor (random sampling, genetic algorithm, kd-tree)
- **Gradient-based** — requires a differentiable model (TensorFlow, PyTorch)

## 1. Imports

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

import dice_ml
from dice_ml.utils import helpers

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load the Dataset

We use the Adult Income dataset from the UCI ML Repository. DiCE ships a helper that loads a cleaned 30k-row subset with 8 features. The outcome (`income`) is binarised: `0` = ≤50K, `1` = >50K.

> **Project note:** Our main app uses the full 47k-row dataset loaded via `ucimlrepo`. Here we use DiCE's built-in helper to stay close to the tutorial and keep this notebook self-contained.

In [30]:
dataset = helpers.load_adult_income_dataset()
dataset.head()

,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,28,Private,Bachelors,Single,White-Collar,White,Female,60,0
1,30,Self-Employed,Assoc,Married,Professional,White,Male,65,1
2,32,Private,Some-college,Married,White-Collar,White,Male,50,0
3,20,Private,Some-college,Single,Service,White,Female,35,0
4,41,Self-Employed,Some-college,Married,White-Collar,White,Male,50,0


In [31]:
print(f"Shape: {dataset.shape}")
print(f"Columns: {list(dataset.columns)}")
print(f"\nClass balance:\n{dataset['income'].value_counts()}")

Shape: (26048, 9)
Columns: ['age', 'workclass', 'education', 'marital_status', 'occupation', 'race', 'gender', 'hours_per_week', 'income']

Class balance:
income
0    19820
1     6228
Name: count, dtype: int64


In [32]:
# DiCE's helper also exposes metadata about the dataset
adult_info = helpers.get_adult_data_info()
adult_info

{'age': 'age',
 'workclass': 'type of industry (Government, Other/Unknown, Private, Self-Employed)',
 'education': 'education level (Assoc, Bachelors, Doctorate, HS-grad, Masters, Prof-school, School, Some-college)',
 'marital_status': 'marital status (Divorced, Married, Separated, Single, Widowed)',
 'occupation': 'occupation (Blue-Collar, Other/Unknown, Professional, Sales, Service, White-Collar)',
 'race': 'white or other race?',
 'gender': 'male or female?',
 'hours_per_week': 'total work hours per week',
 'income': '0 (<=50K) vs 1 (>50K)'}

## 3. Train/Test Split

In [33]:
# Cast to int — DiCE's helper stores the income column as dtype=object (Python
# ints boxed as generic objects). sklearn's type_of_target() returns 'unknown'
# for object arrays even when the values are 0/1, which causes
# RandomForestClassifier to raise:
#   ValueError: Unknown label type: unknown.
# Casting to int64 here means the correct dtype flows into y_train and y_test.
target = dataset["income"].astype(int)

train_dataset, test_dataset, y_train, y_test = train_test_split(
    dataset,
    target,
    test_size=0.2,
    random_state=0,
    stratify=target,
)

x_train = train_dataset.drop("income", axis=1)
x_test  = test_dataset.drop("income", axis=1)

print(f"Train: {x_train.shape}  |  Test: {x_test.shape}")
print(f"y_train dtype: {y_train.dtype}")

Train: (20838, 8)  |  Test: (5210, 8)
y_train dtype: int64


## 4. Create the DiCE Data Object

`dice_ml.Data` wraps the training dataframe and tells DiCE which features are continuous (everything else is treated as categorical). This metadata is used later when generating counterfactuals — DiCE needs to know valid value ranges and types.

In [34]:
d = dice_ml.Data(
    dataframe=train_dataset,
    continuous_features=["age", "hours_per_week"],
    outcome_name="income",
)

## 5. Train a Model

We build a sklearn `Pipeline` that one-hot encodes categorical features and feeds them into a `RandomForestClassifier`. Wrapping everything in a Pipeline is important for DiCE — it needs to call `model.predict()` on raw (pre-encoded) DataFrames, which the Pipeline handles transparently.

In [35]:
numerical   = ["age", "hours_per_week"]
categorical = x_train.columns.difference(numerical)

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical),
])

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   RandomForestClassifier(random_state=42)),
])

model = clf.fit(x_train, y_train)

train_acc = model.score(x_train, y_train)
test_acc  = model.score(x_test,  y_test)
print(f"Train accuracy: {train_acc:.3f}  |  Test accuracy: {test_acc:.3f}")

Train accuracy: 0.832  |  Test accuracy: 0.819


## 6. Create the DiCE Model Object

`dice_ml.Model` wraps the trained model and tells DiCE which backend to use. For sklearn Pipelines, `backend="sklearn"` is required.

In [36]:
m = dice_ml.Model(model=model, backend="sklearn")

## 7. Initialise the DiCE Explainer

`dice_ml.Dice` combines the data object and model object into an explainer. The `method` parameter controls how counterfactuals are searched:

| Method | Description |
|--------|-------------|
| `"random"` | Random sampling from the feature space — fast, less optimised |
| `"genetic"` | Genetic algorithm — slower but finds better (more proximate) CFs |
| `"kdtree"` | KD-tree nearest-neighbour lookup on training data — CFs are guaranteed to be real data points |

In [37]:
exp = dice_ml.Dice(d, m, method="random")

## 8. Generate Counterfactuals

### 8.1 Basic generation

`generate_counterfactuals` takes a query instance (one or more rows from `x_test`) and returns `total_CFs` counterfactual examples that achieve `desired_class`.

`desired_class="opposite"` means: flip the predicted class (0→1 or 1→0).

In [38]:
# Inspect the query instance we're explaining
query_instance = x_test.iloc[0:1]
print("Query instance:")
display(query_instance)
print(f"\nModel prediction: {model.predict(query_instance)[0]} (0=<=50K, 1=>50K)")

Query instance:


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week
14946,29,Private,HS-grad,Married,Blue-Collar,White,Female,38



Model prediction: 0 (0=<=50K, 1=>50K)


In [39]:
e1 = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
)

# show_only_changes=True highlights only the features that differ from the original
e1.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  8.10it/s]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Masters,-,Professional,-,-,-,1
1,-,-,Masters,-,White-Collar,-,-,-,1
2,-,-,Prof-school,-,-,-,Male,-,1


In [40]:
# Full view — all feature values shown
e1.visualize_as_dataframe(show_only_changes=False)

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,Masters,Married,Professional,White,Female,38,1
1,29,Private,Masters,Married,White-Collar,White,Female,38,1
2,29,Private,Prof-school,Married,Blue-Collar,White,Male,38,1


### 8.2 Restrict which features can vary

In practice, some features are immutable (e.g. `age` in a retrospective scenario) or irrelevant to the intervention. `features_to_vary` constrains the search to only modify the listed features.

In [41]:
e2 = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
    features_to_vary=["education", "occupation", "hours_per_week"],
)
e2.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  7.29it/s]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Prof-school,-,White-Collar,-,-,-,1
1,-,-,Assoc,-,White-Collar,-,-,-,1
2,-,-,Prof-school,-,Professional,-,-,-,1


### 8.3 Restrict the permitted range of feature values

`permitted_range` sets bounds per feature — useful for enforcing domain constraints (e.g. age can only increase, or only certain education levels are realistic for a given person).

In [42]:
e3 = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
    permitted_range={
        "age":       [20, 40],
        "education": ["Bachelors", "Masters", "Doctorate", "Prof-school"],
    },
)
e3.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  7.79it/s]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Bachelors,-,-,-,Male,-,1
1,-,Government,Prof-school,-,-,-,-,-,1
2,-,-,Prof-school,-,White-Collar,-,-,-,1


### 8.4 Try a different generation method — genetic algorithm

The genetic algorithm tends to produce counterfactuals that are closer to the original instance (more proximate) by optimising a loss that penalises distance.

In [43]:
exp_genetic = dice_ml.Dice(d, m, method="genetic")

e4 = exp_genetic.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
)
e4.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,30,-,Bachelors,-,Professional,-,-,-,1
0,27,-,Assoc,-,Service,-,-,-,1
0,-,-,Bachelors,-,-,-,Male,40,1


### 8.5 KD-tree method — counterfactuals drawn from real training instances

In [44]:
exp_kdtree = dice_ml.Dice(d, m, method="kdtree")

e5 = exp_kdtree.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
)
e5.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  3.09it/s]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
7496,-,-,Bachelors,-,Professional,-,-,-,1
19163,-,-,Assoc,-,Service,-,-,-,1
16736,-,-,Bachelors,-,-,-,Male,-,1


## 9. Accessing the Raw Counterfactual Data

`visualize_as_dataframe` is convenient for exploration, but for our Dash app we'll need the raw values. The counterfactual DataFrames are accessible directly from the result object.

In [45]:
# The CounterfactualExplanations object contains one CFExamples object per query instance
cf_examples = e1.cf_examples_list[0]

print("Original instance:")
display(cf_examples.test_instance_df)

print("\nCounterfactuals (final_cfs_df):")
display(cf_examples.final_cfs_df)

Original instance:


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Counterfactuals (final_cfs_df):


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,Masters,Married,Professional,White,Female,38,1
1,29,Private,Masters,Married,White-Collar,White,Female,38,1
2,29,Private,Prof-school,Married,Blue-Collar,White,Male,38,1


In [46]:
# Summary of what changed in each counterfactual vs the original
original = cf_examples.test_instance_df.iloc[0]
cfs      = cf_examples.final_cfs_df

print("Features changed per counterfactual:")
for i, row in cfs.iterrows():
    changed = [col for col in original.index if col in row.index and row[col] != original[col]]
    print(f"  CF {i}: {changed}")

Features changed per counterfactual:
  CF 0: ['education', 'occupation', 'income']
  CF 1: ['education', 'occupation', 'income']
  CF 2: ['education', 'gender', 'income']
